# SciRet v2 — Tier 2 Notebook (Bulletproof Edition)
**Works on: Kaggle · Google Colab · Local PC · University Lab PC**

- All heavy steps are cached — restart anytime, never re-compute
- Environment auto-detected — no manual path changes needed
- Run All and walk away — safe to crash and resume

Results saved to `tier2_results/` → zip downloaded at the end.


In [ ]:
# Cell 0 — run this ONCE then comment it out
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "chromadb==0.5.23",
    "rank-bm25==0.2.2",
    "sentence-transformers==3.3.1",
    "google-generativeai==0.8.3",
    "PyMuPDF==1.24.14",
    "ragas==0.2.6",
    "langchain-google-genai",
    "datasets",
], check=False)
print("done — now restart kernel and run from Cell 1")

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

# ── Detect environment ────────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists("/content")
ON_LOCAL  = not ON_KAGGLE and not ON_COLAB

print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local'}")

# ── Working directory ─────────────────────────────────────────────────────────
if ON_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
    # Check if previous saved version is attached as input
    SAVED_INPUT = Path("/kaggle/input/tier-2-50k")
    if not SAVED_INPUT.exists():
        # Try other common names
        for p in Path("/kaggle/input").iterdir() if Path("/kaggle/input").exists() else []:
            if "tier" in p.name.lower() or "sciret" in p.name.lower():
                SAVED_INPUT = p
                break
        else:
            SAVED_INPUT = None
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR    = Path("/content/sciret_tier2")
    SAVED_INPUT = None
else:
    # Local / lab PC — set this to wherever you want output
    WORK_DIR    = Path.home() / "sciret_tier2"
    SAVED_INPUT = None

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working dir  : {WORK_DIR}")
print(f"Saved input  : {SAVED_INPUT if SAVED_INPUT and SAVED_INPUT.exists() else 'None'}")

# ── CORD-19 data path ─────────────────────────────────────────────────────────
if ON_KAGGLE:
    CORD19_ROOT = Path("/kaggle/input/datasets/organizations/allen-institute-for-ai/CORD-19-research-challenge")
    if not CORD19_ROOT.exists():
        # Search for it
        for p in Path("/kaggle/input").iterdir():
            if "cord" in p.name.lower() or "covid" in p.name.lower():
                CORD19_ROOT = p
                break
elif ON_COLAB:
    CORD19_ROOT = Path("/content/drive/MyDrive/CORD-19")
else:
    # Local — change this to your CORD-19 download path
    CORD19_ROOT = Path(os.environ.get("CORD19_PATH", str(Path.home() / "CORD-19")))

print(f"CORD-19 root : {CORD19_ROOT}")
print(f"CORD-19 exists: {CORD19_ROOT.exists()}")


In [ ]:
import nltk
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)


In [ ]:
import torch, numpy as np, pandas as pd, warnings, json, re, pickle
warnings.filterwarnings("ignore")

# ── Gemini API key ────────────────────────────────────────────────────────────
GEMINI_KEY = ""
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        GEMINI_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
        print("Gemini key: loaded from Kaggle secrets")
    except Exception as e:
        print(f"Kaggle secrets failed: {e}")

if not GEMINI_KEY:
    GEMINI_KEY = os.environ.get("GEMINI_API_KEY", "")
    print(f"Gemini key: {'SET from env' if GEMINI_KEY else 'NOT SET — generation will be skipped'}")

os.environ["GEMINI_API_KEY"] = GEMINI_KEY

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    VRAM_GB = 0
    print("WARNING: No GPU — embedding will be slow (~8-12 hours for 50K chunks)")

# ── Models ────────────────────────────────────────────────────────────────────
BGE_MODEL      = "BAAI/bge-m3"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CLIP_MODEL     = "openai/clip-vit-base-patch32"
BLIP2_MODEL    = "Salesforce/blip2-opt-2.7b"
GEMINI_MODEL   = "gemini-1.5-flash"

# ── Hyperparameters ───────────────────────────────────────────────────────────
TIER_SIZE    = 50_000
CHUNK_SIZE   = 400
CHUNK_OVERLAP= 50
MIN_TOKENS   = 30
RRF_K        = 60
STAGE1_K     = 50
RERANK_TOP_K = 5
SEED         = 42
RUN_BLIP2    = True

# Auto batch size based on VRAM
if   VRAM_GB >= 30: EMBED_BATCH = 128
elif VRAM_GB >= 15: EMBED_BATCH = 64
elif VRAM_GB >= 7:  EMBED_BATCH = 32
elif VRAM_GB >= 3:  EMBED_BATCH = 8
else:               EMBED_BATCH = 4

print(f"Batch size : {EMBED_BATCH}")

# ── Directories ───────────────────────────────────────────────────────────────
RESULTS_DIR = WORK_DIR / "tier2_results"
EMBED_DIR   = WORK_DIR / "embeddings"
CHROMA_DIR  = WORK_DIR / "chroma_db"
FIGURES_DIR = WORK_DIR / "figures"
for d in [RESULTS_DIR, EMBED_DIR, CHROMA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Cache paths ───────────────────────────────────────────────────────────────
PAPERS_CACHE  = WORK_DIR / "papers_tier2.parquet"
CHUNKS_CACHE  = WORK_DIR / "chunks_tier2.parquet"
EMBED_CACHE   = EMBED_DIR / "bge_m3_tier2.npy"
IDS_CACHE     = EMBED_DIR / "chunk_ids_tier2.txt"
BM25_CACHE    = EMBED_DIR / "bm25_tier2.pkl"
FIGMETA_CACHE = WORK_DIR / "figures_metadata_tier2.parquet"

# Check if saved input has embeddings
if SAVED_INPUT and SAVED_INPUT.exists():
    saved_npy = SAVED_INPUT / "embeddings" / "bge_m3_tier2.npy"
    saved_ids = SAVED_INPUT / "embeddings" / "chunk_ids_tier2.txt"
    if saved_npy.exists() and not EMBED_CACHE.exists():
        import shutil
        shutil.copy(saved_npy, EMBED_CACHE)
        shutil.copy(saved_ids, IDS_CACHE)
        print(f"Embeddings restored from saved input")
    # Also restore chunks if available
    saved_chunks = SAVED_INPUT / "chunks_tier2.parquet"
    if saved_chunks.exists() and not CHUNKS_CACHE.exists():
        shutil.copy(saved_chunks, CHUNKS_CACHE)
        print("Chunks restored from saved input")

# ── Tier 1 baseline (hardcoded from local run) ────────────────────────────────
TIER1 = {
    "recall": {
        1:  {"dense":0.104,"bm25":0.139,"hybrid":0.124},
        3:  {"dense":0.290,"bm25":0.321,"hybrid":0.378},
        5:  {"dense":0.447,"bm25":0.433,"hybrid":0.568},
        10: {"dense":0.670,"bm25":0.581,"hybrid":0.950},
        20: {"dense":0.855,"bm25":0.778,"hybrid":1.000},
    },
    "precision_no_rerank": {1:0.950,3:0.917,5:0.870,10:0.805},
    "precision_rerank":    {1:0.850,3:0.800,5:0.700,10:0.575},
    "context_recall_dpr":    0.503,
    "context_recall_sciret": 0.586,
    "faithfulness":          0.904,
}

print("Config ready")


## Step 1 — Load CORD-19 and sample 50K papers

In [ ]:
if PAPERS_CACHE.exists():
    print(f"CACHE HIT — loading papers from {PAPERS_CACHE}")
    papers = pd.read_parquet(PAPERS_CACHE)
    print(f"Loaded {len(papers):,} papers")
else:
    print("CACHE MISS — loading from CORD-19 metadata.csv ...")
    meta_path = CORD19_ROOT / "metadata.csv"
    if not meta_path.exists():
        raise FileNotFoundError(
            f"metadata.csv not found at {meta_path}\n"
            "Download CORD-19 from https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge"
        )
    meta = pd.read_csv(meta_path, low_memory=False)
    print(f"Total rows: {len(meta):,}")

    meta["publish_time_parsed"] = pd.to_datetime(meta["publish_time"], errors="coerce")
    meta["publish_year"] = meta["publish_time_parsed"].dt.year

    meta_f = meta[
        meta["abstract"].notna() &
        (meta["abstract"].str.len() > 50) &
        meta["title"].notna()
    ].copy()
    print(f"After filtering: {len(meta_f):,} papers")

    def stratified_sample(df, n, seed=SEED):
        if n >= len(df): return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        year_col = df["publish_year"].fillna(-1).astype(int)
        groups = df.groupby(year_col, group_keys=False)
        sizes = (groups.size() / len(df) * n).round().astype(int).clip(lower=1)
        while sizes.sum() > n: sizes[sizes.idxmax()] -= 1
        while sizes.sum() < n: sizes[sizes.idxmax()] += 1
        parts = [g.sample(n=min(sizes[y], len(g)), random_state=seed) for y, g in groups]
        return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    papers = stratified_sample(meta_f, TIER_SIZE, seed=SEED)
    KEEP = ["cord_uid","title","abstract","authors","journal",
            "publish_time","publish_year","doi","pmcid","pdf_json_files"]
    papers = papers[[c for c in KEEP if c in papers.columns]].reset_index(drop=True)
    papers.to_parquet(PAPERS_CACHE, index=False)
    print(f"Saved {len(papers):,} papers → {PAPERS_CACHE}")

print(f"Year distribution (top 5):")
print(papers["publish_year"].value_counts().sort_index().tail(6).to_string())


## Step 2 — Text chunking (sentence window)

In [ ]:
if CHUNKS_CACHE.exists():
    print(f"CACHE HIT — loading chunks from {CHUNKS_CACHE}")
    chunks = pd.read_parquet(CHUNKS_CACHE)
    print(f"Loaded {len(chunks):,} chunks")
else:
    print("CACHE MISS — building chunks ...")
    from nltk.tokenize import sent_tokenize

    def sentence_window_chunks(text, max_tokens=CHUNK_SIZE,
                               overlap_tokens=CHUNK_OVERLAP, min_tokens=MIN_TOKENS):
        if not text or len(text.strip()) < 20: return []
        sentences = sent_tokenize(text)
        chunks, current, current_len = [], [], 0
        for sent in sentences:
            t = len(sent.split())
            if current_len + t > max_tokens and current:
                chunk_text = " ".join(current)
                if len(chunk_text.split()) >= min_tokens:
                    chunks.append(chunk_text)
                overlap, overlap_len = [], 0
                for s in reversed(current):
                    st = len(s.split())
                    if overlap_len + st <= overlap_tokens:
                        overlap.insert(0, s); overlap_len += st
                    else: break
                current, current_len = overlap, overlap_len
            current.append(sent); current_len += t
        if current:
            chunk_text = " ".join(current)
            if len(chunk_text.split()) >= min_tokens:
                chunks.append(chunk_text)
        return chunks

    records = []
    for _, row in papers.iterrows():
        cid   = str(row.get("cord_uid", "unknown"))
        title = (row.get("title") or "").strip()
        text  = f"{title}. {(row.get('abstract') or '')}".strip()
        for idx, chunk in enumerate(sentence_window_chunks(text)):
            records.append({
                "chunk_id":     f"{cid}_c{idx:03d}",
                "cord_uid":     cid,
                "chunk_index":  idx,
                "title":        title,
                "chunk_text":   chunk,
                "n_tokens":     len(chunk.split()),
                "journal":      (row.get("journal") or ""),
                "publish_time": (row.get("publish_time") or ""),
                "doi":          (row.get("doi") or ""),
            })

    chunks = pd.DataFrame(records)
    chunks.to_parquet(CHUNKS_CACHE, index=False)
    print(f"Built {len(chunks):,} chunks — saved to {CHUNKS_CACHE}")

chunks = chunks.reset_index(drop=True)
# Drop duplicate chunk_ids keeping first occurrence
dupes = chunks["chunk_id"].duplicated().sum()
if dupes > 0:
    print(f"Dropping {dupes} duplicate chunk_ids")
    chunks = chunks.drop_duplicates(subset="chunk_id", keep="first").reset_index(drop=True)

chunk_lookup = {row["chunk_id"]: row.to_dict() 
                for _, row in chunks.iterrows()}
print(f"chunk_lookup built: {len(chunk_lookup):,} entries")
print(f"Mean tokens: {chunks['n_tokens'].mean():.1f}")
print(f"P95 tokens : {chunks['n_tokens'].quantile(0.95):.1f}")
chunk_ids = chunks["chunk_id"].tolist()
print(f"chunk_ids synced: {len(chunk_ids):,}")

## Step 3 — BGE-M3 embeddings (~45-80 min first run, instant after)

In [ ]:
from sentence_transformers import SentenceTransformer

if EMBED_CACHE.exists() and IDS_CACHE.exists():
    print(f"CACHE HIT — loading embeddings from disk")
    vecs      = np.load(str(EMBED_CACHE))
    chunk_ids = open(IDS_CACHE).read().splitlines()
    print(f"Loaded: shape={vecs.shape}  IDs={len(chunk_ids):,}")
else:
    print(f"CACHE MISS — computing BGE-M3 embeddings (batch={EMBED_BATCH}) ...")
    embedder = SentenceTransformer(BGE_MODEL, device=DEVICE)
    embedder.max_seq_length = 512
    print(f"BGE-M3 loaded on {DEVICE.upper()}")

    texts     = chunks["chunk_text"].fillna("").tolist()
    chunk_ids = chunks["chunk_id"].tolist()
    vecs_list = []
    t0        = time.time()

    for i in range(0, len(texts), EMBED_BATCH):
        batch = texts[i:i+EMBED_BATCH]
        v     = embedder.encode(batch, normalize_embeddings=True,
                                show_progress_bar=False)
        vecs_list.append(v)
        done = i + len(batch)
        if done % 3200 == 0 or done == len(texts):
            elapsed = time.time() - t0
            eta     = elapsed / done * (len(texts) - done)
            print(f"  {done:,}/{len(texts):,}  "
                  f"elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m")

    vecs = np.vstack(vecs_list)
    elapsed = time.time() - t0
    print(f"Complete: shape={vecs.shape}  time={elapsed/60:.1f}min")

    # Save as numpy — fast, no column loop
    np.save(str(EMBED_CACHE), vecs)
    with open(IDS_CACHE, "w") as f:
        f.write("\n".join(chunk_ids))
    print(f"Saved → {EMBED_CACHE}")
    print("SAVE A KAGGLE VERSION NOW before continuing!")

    del embedder
    if DEVICE == "cuda": torch.cuda.empty_cache()

print(f"Embeddings ready: {vecs.shape}")


## Step 4 — ChromaDB index

In [ ]:
import chromadb

COL_NAME = f"sciret_tier2_bge_m3_cs{CHUNK_SIZE}_o{CHUNK_OVERLAP}"
client   = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Check if collection already exists and is fully populated
try:
    col = client.get_collection(COL_NAME)
    if col.count() == len(chunk_ids):
        print(f"CACHE HIT — ChromaDB collection '{COL_NAME}': {col.count():,} chunks")
    else:
        print(f"Partial collection ({col.count():,}/{len(chunk_ids):,}) — rebuilding")
        client.delete_collection(COL_NAME)
        raise ValueError("rebuild")
except Exception:
    print(f"CACHE MISS — building ChromaDB index ...")
    try: client.delete_collection(COL_NAME)
    except: pass

    col       = client.create_collection(COL_NAME, metadata={"hnsw:space":"cosine"})
    BATCH     = 512
    meta_list = chunks[["title","journal","cord_uid"]].fillna("").to_dict("records")

    for i in range(0, len(chunk_ids), BATCH):
        col.add(
            ids        = chunk_ids[i:i+BATCH],
            embeddings = vecs[i:i+BATCH].tolist(),
            metadatas  = meta_list[i:i+BATCH],
        )
        if (i // BATCH + 1) % 20 == 0:
            print(f"  indexed {min(i+BATCH,len(chunk_ids)):,}/{len(chunk_ids):,}")

    print(f"ChromaDB collection '{COL_NAME}': {col.count():,} chunks")


## Step 5 — BM25 index

In [ ]:
from rank_bm25 import BM25Okapi

if BM25_CACHE.exists():
    print(f"CACHE HIT — loading BM25 from {BM25_CACHE}")
    with open(BM25_CACHE, "rb") as f:
        bm25_data = pickle.load(f)
    bm25     = bm25_data["index"]
    bm25_ids = bm25_data["chunk_ids"]
    print(f"BM25 loaded: {len(bm25_ids):,} docs")
else:
    print("CACHE MISS — building BM25 index ...")
    corpus_tokens = [t.lower().split()
                     for t in chunks["chunk_text"].fillna("").tolist()]
    bm25     = BM25Okapi(corpus_tokens)
    bm25_ids = chunk_ids
    with open(BM25_CACHE, "wb") as f:
        pickle.dump({"index": bm25, "chunk_ids": bm25_ids}, f)
    print(f"BM25 built and saved: {len(bm25_ids):,} docs")


## Step 6 — Figure extraction from PDFs

In [ ]:
import fitz
from PIL import Image as PILImage

CAPTION_RE = re.compile(
    r"(fig(?:ure)?\.?\s*\d+[a-z]?\s*[:.\-]?\s*.{0,400})",
    flags=re.IGNORECASE | re.DOTALL)

if FIGMETA_CACHE.exists():
    print(f"CACHE HIT — loading figure metadata from {FIGMETA_CACHE}")
    df_fig = pd.read_parquet(FIGMETA_CACHE)
    print(f"Loaded {len(df_fig):,} figures")
else:
    print("CACHE MISS — extracting figures from PDFs ...")

    def nearest_caption(page_text, img_idx):
        matches = CAPTION_RE.findall(page_text or "")
        if not matches: return ""
        m = matches[min(img_idx, len(matches)-1)]
        return " ".join(m.split())[:500]

    def extract_pdf(pdf_path, out_dir):
        records, cord_uid = [], pdf_path.stem
        try: doc = fitz.open(pdf_path)
        except: return records
        out_dir.mkdir(parents=True, exist_ok=True)
        fig_n = 0
        for page_idx, page in enumerate(doc):
            page_text = page.get_text("text")
            for img_idx, img in enumerate(page.get_images(full=True)):
                try:
                    pix = fitz.Pixmap(doc, img[0])
                    if pix.n > 4: pix = fitz.Pixmap(fitz.csRGB, pix)
                    if pix.width < 100 or pix.height < 100: continue
                    fid   = f"{cord_uid}_fig_{fig_n:03d}"
                    asset = out_dir / f"{fid}.png"
                    pix.save(str(asset))
                    records.append({"figure_id": fid, "cord_uid": cord_uid,
                                    "page": page_idx, "asset_path": str(asset),
                                    "caption_raw": nearest_caption(page_text, img_idx),
                                    "width": pix.width, "height": pix.height})
                    fig_n += 1
                except: continue
        return records

    PDF_ROOT      = CORD19_ROOT / "document_parses"
    sampled_uids  = set(papers["cord_uid"].tolist())
    pdf_paths     = [p for p in PDF_ROOT.rglob("*.pdf")
                     if p.stem in sampled_uids] if PDF_ROOT.exists() else []
    print(f"PDFs found: {len(pdf_paths):,}")

    figure_records = []
    for i, pdf in enumerate(pdf_paths):
        figure_records.extend(extract_pdf(pdf, FIGURES_DIR / pdf.stem))
        if (i+1) % 100 == 0:
            print(f"  {i+1}/{len(pdf_paths)} PDFs → {len(figure_records):,} figures")

    df_fig = pd.DataFrame(figure_records) if figure_records else pd.DataFrame(
        columns=["figure_id","cord_uid","page","asset_path",
                 "caption_raw","width","height"])
    df_fig.to_parquet(FIGMETA_CACHE, index=False)
    print(f"Saved {len(df_fig):,} figures → {FIGMETA_CACHE}")


## Step 7 — BLIP-2 captioning

In [ ]:
df_fig["caption"] = df_fig.get("caption", df_fig.get("caption_raw", pd.Series([""] * len(df_fig)))).fillna("").str.strip()

blip_done = "caption" in df_fig.columns and (df_fig["caption"].str.startswith("[blip2]")).any()

if RUN_BLIP2 and len(df_fig) > 0 and not blip_done:
    print("Running BLIP-2 captioning ...")
    from transformers import Blip2Processor, Blip2ForConditionalGeneration
    proc = Blip2Processor.from_pretrained(BLIP2_MODEL)
    blip = Blip2ForConditionalGeneration.from_pretrained(
        BLIP2_MODEL, torch_dtype=torch.float16).to(DEVICE).eval()

    need = df_fig.index[df_fig["caption"].str.len() < 30].tolist()[:500]
    print(f"Figures needing captions: {len(need):,}")
    for i, idx in enumerate(need):
        try:
            img    = PILImage.open(df_fig.loc[idx,"asset_path"]).convert("RGB")
            inputs = proc(images=img, return_tensors="pt").to(DEVICE, torch.float16)
            out    = blip.generate(**inputs, max_new_tokens=40)
            cap    = proc.decode(out[0], skip_special_tokens=True).strip()
            df_fig.at[idx,"caption"] = f"[blip2] {cap}"
        except: pass
        if (i+1) % 50 == 0: print(f"  {i+1}/{len(need)}")

    del blip, proc
    torch.cuda.empty_cache()
    df_fig.to_parquet(FIGMETA_CACHE, index=False)
    print("BLIP-2 done — cache updated")
elif blip_done:
    print("CACHE HIT — BLIP-2 captions already in metadata")
else:
    print(f"Skipping BLIP-2 ({len(df_fig)} figures available)")

df_fig["text_for_embedding"] = df_fig.get("caption", pd.Series([""] * len(df_fig))).fillna("").str[:500]


## Step 8 — CLIP figure embeddings

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_col_name = "sciret_tier2_clip_figures"
try:
    fig_col = client.get_collection(clip_col_name)
    print(f"CACHE HIT — CLIP collection: {fig_col.count():,} figures")
    clip_cached = True
except:
    clip_cached = False

# Always load CLIP model for query-time use
clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained(CLIP_MODEL)

# Fix for newer HF versions returning ModelOutput instead of tensor
def clip_text_embed(text_list):
    with torch.no_grad():
        t_in = clip_proc(text=text_list, return_tensors="pt",
                         padding=True, truncation=True).to(DEVICE)
        out  = clip_model.get_text_features(**t_in)
        vec  = (out if isinstance(out, torch.Tensor) else out.pooler_output)
        return vec.cpu().numpy()

def clip_image_embed(img):
    with torch.no_grad():
        inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
        out    = clip_model.get_image_features(**inputs)
        vec    = (out if isinstance(out, torch.Tensor) else out.pooler_output)
        return vec.cpu().numpy()

# Sanity check
v = clip_text_embed(["covid lung CT"])
print(f"CLIP ok — dim={v.shape[1]}")

if not clip_cached and len(df_fig) > 0:
    print("Building CLIP figure index ...")
    try: client.delete_collection(clip_col_name)
    except: pass
    fig_col  = client.create_collection(clip_col_name,
                                        metadata={"hnsw:space":"cosine"})
    fig_vecs, fig_ids = [], []
    for _, row in df_fig.iterrows():
        try:
            img = PILImage.open(row["asset_path"]).convert("RGB")
            vec = clip_image_embed(img)
            fig_vecs.append(vec[0].tolist())
            fig_ids.append(row["figure_id"])
        except: continue
    if fig_vecs:
        fig_col.add(ids=fig_ids, embeddings=fig_vecs,
                    metadatas=[{"cord_uid": fid.rsplit("_fig_",1)[0]}
                               for fid in fig_ids])
        print(f"Figure collection: {fig_col.count():,} indexed")
else:
    try: fig_col = client.get_collection(clip_col_name)
    except:
        fig_col = client.create_collection(clip_col_name,
                                           metadata={"hnsw:space":"cosine"})
    if fig_col.count() == 0:
        print("No figures available — text-only pipeline")

del clip_model
if DEVICE == "cuda": torch.cuda.empty_cache()
print("CLIP ready")


## Step 9 — Evaluation questions (50)

In [ ]:
EVAL_QUESTIONS = [
    {"q":"What CT imaging findings are characteristic of COVID-19 pneumonia?","type":"imaging"},
    {"q":"How does chest X-ray appearance differ between COVID-19 and bacterial pneumonia?","type":"imaging"},
    {"q":"What ultrasound features are observed in COVID-19 patients?","type":"imaging"},
    {"q":"How does CT severity score correlate with COVID-19 outcomes?","type":"imaging"},
    {"q":"What are the radiological features of COVID-19 in children?","type":"imaging"},
    {"q":"How does ground-glass opacity evolve over the course of COVID-19?","type":"imaging"},
    {"q":"What MRI findings have been reported in COVID-19 neurological complications?","type":"imaging"},
    {"q":"How is CT used to monitor COVID-19 treatment response?","type":"imaging"},
    {"q":"What are the pulmonary sequelae visible on CT after COVID-19 recovery?","type":"imaging"},
    {"q":"What deep learning approaches have been applied to COVID-19 CT diagnosis?","type":"imaging"},
    {"q":"How does SARS-CoV-2 spike protein bind to ACE2 receptors?","type":"molecular"},
    {"q":"What is the role of the furin cleavage site in SARS-CoV-2 infection?","type":"molecular"},
    {"q":"How does SARS-CoV-2 evade the innate immune response?","type":"molecular"},
    {"q":"What mutations in the spike protein affect vaccine efficacy?","type":"molecular"},
    {"q":"How does TMPRSS2 facilitate SARS-CoV-2 cell entry?","type":"molecular"},
    {"q":"What is the mechanism of SARS-CoV-2 RNA replication?","type":"molecular"},
    {"q":"How do neutralising antibodies target the SARS-CoV-2 receptor binding domain?","type":"molecular"},
    {"q":"What is the role of the nucleocapsid protein in SARS-CoV-2?","type":"molecular"},
    {"q":"How does SARS-CoV-2 affect the renin-angiotensin system?","type":"molecular"},
    {"q":"What structural features distinguish SARS-CoV-2 from SARS-CoV-1?","type":"molecular"},
    {"q":"What are the risk factors for severe COVID-19 disease?","type":"clinical"},
    {"q":"What role does IL-6 play in COVID-19 cytokine storm?","type":"clinical"},
    {"q":"What are the cardiac complications associated with COVID-19?","type":"clinical"},
    {"q":"How is long COVID defined and what are its most common symptoms?","type":"clinical"},
    {"q":"What are the neurological complications of COVID-19?","type":"clinical"},
    {"q":"How does COVID-19 affect patients with diabetes?","type":"clinical"},
    {"q":"What is the mortality rate of COVID-19 in ICU patients?","type":"clinical"},
    {"q":"How does age affect COVID-19 severity and outcomes?","type":"clinical"},
    {"q":"What are the renal complications of COVID-19?","type":"clinical"},
    {"q":"What coagulation abnormalities are observed in severe COVID-19?","type":"clinical"},
    {"q":"How effective are mRNA vaccines against COVID-19 variants?","type":"treatment"},
    {"q":"What antiviral drugs have shown efficacy against SARS-CoV-2?","type":"treatment"},
    {"q":"What is the evidence for dexamethasone in severe COVID-19?","type":"treatment"},
    {"q":"How effective is remdesivir for COVID-19 treatment?","type":"treatment"},
    {"q":"What is the role of convalescent plasma in COVID-19 treatment?","type":"treatment"},
    {"q":"How do monoclonal antibodies work against SARS-CoV-2?","type":"treatment"},
    {"q":"What ventilation strategies are used for COVID-19 ARDS?","type":"treatment"},
    {"q":"What is the evidence for prone positioning in COVID-19?","type":"treatment"},
    {"q":"How effective are COVID-19 vaccines in immunocompromised patients?","type":"treatment"},
    {"q":"What is the evidence for anticoagulation therapy in COVID-19?","type":"treatment"},
    {"q":"How was the CORD-19 dataset constructed and what does it contain?","type":"dataset"},
    {"q":"What NLP methods have been applied to COVID-19 literature?","type":"dataset"},
    {"q":"How have information retrieval systems been used to search COVID-19 research?","type":"dataset"},
    {"q":"What are the main limitations of COVID-19 observational studies?","type":"dataset"},
    {"q":"How have preprints affected COVID-19 research dissemination?","type":"dataset"},
    {"q":"How do socioeconomic factors affect COVID-19 outcomes?","type":"synthesis"},
    {"q":"What is the relationship between air pollution and COVID-19 severity?","type":"synthesis"},
    {"q":"How has COVID-19 affected mental health at a population level?","type":"synthesis"},
    {"q":"What lessons from previous coronavirus outbreaks apply to COVID-19?","type":"synthesis"},
    {"q":"How has COVID-19 impacted healthcare delivery and access?","type":"synthesis"},
]
from collections import Counter
print(f"Evaluation set: {len(EVAL_QUESTIONS)} questions")
for t, c in Counter(q['type'] for q in EVAL_QUESTIONS).items():
    print(f"  {t:12s}: {c}")


## Step 10 — Retrieval pipeline

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer(BGE_MODEL, device=DEVICE)
embedder.max_seq_length = 512
reranker = CrossEncoder(RERANKER_MODEL, max_length=512)
print("Embedder and reranker loaded")

def rrf_fuse(ranked_lists, k=RRF_K):
    scores = {}
    for rl in ranked_lists:
        for rank, doc_id in enumerate(rl, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def retrieve_text(query, use_reranker=False, top_k=RERANK_TOP_K):
    q_emb  = embedder.encode([query], normalize_embeddings=True)
    dense  = col.query(query_embeddings=q_emb.tolist(),
                       n_results=STAGE1_K, include=["metadatas"])
    d_ids  = dense["ids"][0]
    toks   = query.lower().split()
    b_scr  = bm25.get_scores(toks)
    b_ids  = [bm25_ids[i] for i in np.argsort(b_scr)[::-1][:STAGE1_K]]
    fused  = rrf_fuse([d_ids, b_ids])
    cands  = []
    for cid, rrf_score in fused[:STAGE1_K]:
        if cid in chunk_lookup:
            row = chunk_lookup[cid]
            cands.append({"chunk_id":cid, "title":row.get("title",""),
                          "chunk_text":row.get("chunk_text",""),
                          "journal":row.get("journal",""),
                          "rrf_score":rrf_score, "rerank_score":None})
    if use_reranker and cands:
        pairs  = [(query, c["chunk_text"]) for c in cands]
        scores = reranker.predict(pairs, show_progress_bar=False)
        for c, s in zip(cands, scores): c["rerank_score"] = float(s)
        cands.sort(key=lambda x: x["rerank_score"], reverse=True)
    return cands[:top_k]

def retrieve_figures(query, top_k=3):
    try:
        if fig_col.count() == 0: return []
    except: return []
    clip_model_q = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
    clip_proc_q  = CLIPProcessor.from_pretrained(CLIP_MODEL)
    with torch.no_grad():
        t_in = clip_proc_q(text=[query], return_tensors="pt",
                          padding=True, truncation=True).to(DEVICE)
        out  = clip_model_q.get_text_features(**t_in)
        q_vec = (out if isinstance(out, torch.Tensor) else out.pooler_output)
        q_vec = q_vec.cpu().numpy()
    del clip_model_q, clip_proc_q
    results = fig_col.query(query_embeddings=q_vec.tolist(),
                            n_results=top_k, include=["metadatas","distances"])
    return [{"figure_id":fid,"distance":dist}
            for fid, dist in zip(results["ids"][0], results["distances"][0])]

print("Retrieval functions ready")


## Step 11 — Recall@K ablation

In [ ]:
recall_cache = RESULTS_DIR / "recall_at_k_tier2.csv"
K_VALUES    = [1, 3, 5, 10, 20]

def recall_at_k(retrieved_ids, relevant_ids, k):
    return len(set(retrieved_ids[:k]) & set(relevant_ids)) / len(relevant_ids) if relevant_ids else 0.0

if recall_cache.exists():
    print(f"CACHE HIT — loading recall results")
    recall_df = pd.read_csv(recall_cache)
else:
    print("Running retrieval ablation (50 queries) ...")
    recall_rows = []
    for qi, item in enumerate(EVAL_QUESTIONS):
        q      = item["q"]
        q_emb  = embedder.encode([q], normalize_embeddings=True)
        d_ids  = col.query(query_embeddings=q_emb.tolist(),
                           n_results=20, include=["metadatas"])["ids"][0]
        toks   = q.lower().split()
        b_scr  = bm25.get_scores(toks)
        b_ids  = [bm25_ids[i] for i in np.argsort(b_scr)[::-1][:20]]
        h_ids  = [cid for cid, _ in rrf_fuse([d_ids, b_ids])]
        relevant = h_ids[:3]
        row = {"query":q, "type":item["type"]}
        for k in K_VALUES:
            row[f"dense_r{k}"]  = recall_at_k(d_ids,  relevant, k)
            row[f"bm25_r{k}"]   = recall_at_k(b_ids,  relevant, k)
            row[f"hybrid_r{k}"] = recall_at_k(h_ids,  relevant, k)
        recall_rows.append(row)
        if (qi+1) % 10 == 0: print(f"  {qi+1}/{len(EVAL_QUESTIONS)} done")
    recall_df = pd.DataFrame(recall_rows)
    recall_df.to_csv(recall_cache, index=False)
    print(f"Saved → {recall_cache}")

print("\n=== TIER 2 RECALL@K ===")
summary_t2 = {}
for k in K_VALUES:
    summary_t2[k] = {
        "dense":  recall_df[f"dense_r{k}"].mean(),
        "bm25":   recall_df[f"bm25_r{k}"].mean(),
        "hybrid": recall_df[f"hybrid_r{k}"].mean(),
    }
    print(f"@{k:2d}  dense={summary_t2[k]['dense']:.3f}  "
          f"bm25={summary_t2[k]['bm25']:.3f}  hybrid={summary_t2[k]['hybrid']:.3f}")

print("\n=== TIER 1 vs TIER 2 (Hybrid) ===")
for k in K_VALUES:
    t1 = TIER1["recall"][k]["hybrid"]
    t2 = summary_t2[k]["hybrid"]
    print(f"@{k:2d}  T1={t1:.3f}  T2={t2:.3f}  delta={t2-t1:+.3f}")


## Step 12 — Reranking effect at Tier 2

In [ ]:
rerank_cache = RESULTS_DIR / "reranking_tier2.csv"

if rerank_cache.exists():
    print("CACHE HIT — loading reranking results")
    rerank_df = pd.read_csv(rerank_cache)
else:
    print("Running reranking evaluation ...")
    rerank_rows = []
    for qi, item in enumerate(EVAL_QUESTIONS):
        q              = item["q"]
        cands_hybrid   = retrieve_text(q, use_reranker=False, top_k=10)
        cands_reranked = retrieve_text(q, use_reranker=True,  top_k=10)
        relevant       = [c["chunk_id"] for c in cands_hybrid[:3]]
        row = {"query":q, "type":item["type"]}
        for k in [1,3,5,10]:
            row[f"p_no_rerank_{k}"] = recall_at_k(
                [c["chunk_id"] for c in cands_hybrid[:k]],   relevant, k)
            row[f"p_rerank_{k}"]    = recall_at_k(
                [c["chunk_id"] for c in cands_reranked[:k]], relevant, k)
        rerank_rows.append(row)
        if (qi+1) % 10 == 0: print(f"  {qi+1}/{len(EVAL_QUESTIONS)} done")
    rerank_df = pd.DataFrame(rerank_rows)
    rerank_df.to_csv(rerank_cache, index=False)

print("\n=== TIER 2 RERANKING ===")
for k in [1,3,5,10]:
    nr = rerank_df[f"p_no_rerank_{k}"].mean()
    r  = rerank_df[f"p_rerank_{k}"].mean()
    t1 = TIER1["precision_no_rerank"][k]
    print(f"P@{k}: no_rerank={nr:.3f}  rerank={r:.3f}  "
          f"delta={r-nr:+.3f}  (T1 delta={TIER1['precision_rerank'][k]-t1:+.3f})")


## Step 13 — RAGAS evaluation with Gemini

In [ ]:
ragas_cache = RESULTS_DIR / "ragas_tier2.csv"
runs_cache  = RESULTS_DIR / "eval_runs_tier2.parquet"

if ragas_cache.exists():
    print("CACHE HIT — loading RAGAS results")
    ragas_df = pd.read_csv(ragas_cache, index_col=0)
    print(ragas_df.to_string())
elif not GEMINI_KEY:
    print("Skipping RAGAS — GEMINI_API_KEY not set")
    ragas_df = None
else:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_KEY)
    llm = genai.GenerativeModel(GEMINI_MODEL)

    SYSTEM_PROMPT = (
        "You are SciRet, a scientific assistant for COVID-19 research. "
        "Answer ONLY using the provided context passages. "
        "Do not use outside knowledge. Cite passages as [1],[2] etc. "
        "Write clearly. Answer in 4-6 sentences."
    )

    def generate(query, passages):
        ctx = "\n\n".join(
            f"[{i}] From: \"{p['title']}\"\n{p['chunk_text'][:500]}"
            for i, p in enumerate(passages, 1))
        try:
            return llm.generate_content(
                f"{SYSTEM_PROMPT}\n\nCONTEXT:\n{ctx}\n\nQUESTION: {query}\n\nANSWER:"
            ).text.strip()
        except Exception as e:
            return f"[error: {e}]"

    print("Generating answers for 50 questions × 3 systems ...")
    eval_records = []
    for qi, item in enumerate(EVAL_QUESTIONS):
        q = item["q"]
        q_emb = embedder.encode([q], normalize_embeddings=True)
        dense_ids = col.query(query_embeddings=q_emb.tolist(),
                              n_results=5, include=["metadatas"])["ids"][0]
        dense_passages = [{"title": chunk_lookup.get(cid,{}).get("title",""),
                           "chunk_text": chunk_lookup.get(cid,{}).get("chunk_text","")}
                          for cid in dense_ids]
        text_passages  = retrieve_text(q, use_reranker=False)
        multi_passages = retrieve_text(q, use_reranker=False)

        for sys_name, passages in [
            ("dpr_baseline", dense_passages),
            ("sciret_text",  text_passages),
            ("sciret_multi", multi_passages),
        ]:
            answer = generate(q, passages)
            eval_records.append({
                "question":   q,
                "query_type": item["type"],
                "system":     sys_name,
                "answer":     answer,
                "contexts":   [p["chunk_text"] for p in passages],
                "ground_truth": answer,
            })
        if (qi+1) % 5 == 0: print(f"  {qi+1}/{len(EVAL_QUESTIONS)} done")

    eval_df = pd.DataFrame(eval_records)
    eval_df.to_parquet(runs_cache, index=False)
    print(f"Saved {len(eval_df):,} eval runs")

    # RAGAS scoring
    try:
        from ragas import evaluate
        from ragas.metrics import (faithfulness, answer_relevancy,
                                   context_precision, context_recall)
        from datasets import Dataset
        from langchain_google_genai import ChatGoogleGenerativeAI
        from ragas.llms import LangchainLLMWrapper

        ragas_llm = LangchainLLMWrapper(
            ChatGoogleGenerativeAI(model="gemini-1.5-flash",
                                   google_api_key=GEMINI_KEY))
        ragas_results = {}
        for sys_name in ["dpr_baseline","sciret_text","sciret_multi"]:
            subset = eval_df[eval_df["system"]==sys_name]
            ds = Dataset.from_dict({
                "question":    subset["question"].tolist(),
                "answer":      subset["answer"].tolist(),
                "contexts":    subset["contexts"].tolist(),
                "ground_truth":subset["ground_truth"].tolist(),
            })
            try:
                result = evaluate(ds,
                    metrics=[faithfulness, answer_relevancy,
                             context_precision, context_recall],
                    llm=ragas_llm)
                ragas_results[sys_name] = dict(result)
            except Exception as e:
                print(f"RAGAS failed for {sys_name}: {e}")
                ragas_results[sys_name] = {}

        ragas_df = pd.DataFrame(ragas_results).T
        ragas_df.to_csv(ragas_cache)
        print("\n=== TIER 2 RAGAS RESULTS ===")
        print(ragas_df.to_string())
    except Exception as e:
        print(f"RAGAS scoring error: {e}")
        ragas_df = None


## Step 14 — Scale comparison and hypothesis testing

In [ ]:
from scipy import stats

print("=" * 60)
print("SCALE COMPARISON: TIER 1 vs TIER 2")
print("=" * 60)

t1_bm25_r1  = TIER1["recall"][1]["bm25"]
t1_dense_r1 = TIER1["recall"][1]["dense"]
t2_bm25_r1  = recall_df["bm25_r1"].mean()
t2_dense_r1 = recall_df["dense_r1"].mean()

print(f"\nH1 — BM25 vs Dense at Recall@1:")
print(f"  Tier 1: BM25={t1_bm25_r1:.3f} Dense={t1_dense_r1:.3f} "
      f"advantage={t1_bm25_r1-t1_dense_r1:+.3f}")
print(f"  Tier 2: BM25={t2_bm25_r1:.3f} Dense={t2_dense_r1:.3f} "
      f"advantage={t2_bm25_r1-t2_dense_r1:+.3f}")
h1 = "NARROWED/REVERSED" if (t2_bm25_r1-t2_dense_r1)<(t1_bm25_r1-t1_dense_r1) else "PERSISTS"
print(f"  H1 verdict: {h1}")

t1_d5 = TIER1["precision_rerank"][5] - TIER1["precision_no_rerank"][5]
t2_d5 = rerank_df["p_rerank_5"].mean() - rerank_df["p_no_rerank_5"].mean()
print(f"\nH2 — Reranking P@5: T1={t1_d5:+.3f}  T2={t2_d5:+.3f}")
print(f"  H2 verdict: {'DOMAIN MISMATCH DOMINATES' if t2_d5 < 0 else 'SCALE WAS THE FACTOR'}")

t2_h10 = recall_df["hybrid_r10"].tolist()
t2_d10 = recall_df["dense_r10"].tolist()
stat, pval = stats.wilcoxon(t2_h10, t2_d10)
print(f"\nHybrid vs Dense Recall@10 — Wilcoxon p={pval:.4f} "
      f"{'SIGNIFICANT' if pval<0.05 else 'NOT SIGNIFICANT'}")

summary = pd.DataFrame({
    "tier":             ["tier1","tier2"],
    "hybrid_recall_10": [TIER1["recall"][10]["hybrid"], recall_df["hybrid_r10"].mean()],
    "dense_recall_10":  [TIER1["recall"][10]["dense"],  recall_df["dense_r10"].mean()],
    "bm25_recall_10":   [TIER1["recall"][10]["bm25"],   recall_df["bm25_r10"].mean()],
    "rerank_delta_p5":  [t1_d5, t2_d5],
    "bm25_r1_advantage":[t1_bm25_r1-t1_dense_r1, t2_bm25_r1-t2_dense_r1],
})
summary.to_csv(RESULTS_DIR / "scale_comparison.csv", index=False)
print("\nscale_comparison.csv saved")
print(summary.to_string(index=False))


## Step 15 — Export passages for manual annotation

In [ ]:
# Export top-10 passages per query for manual relevance annotation
# Send this CSV to Collaborator A for annotation
annotation_rows = []
for item in EVAL_QUESTIONS:
    q      = item["q"]
    q_emb  = embedder.encode([q], normalize_embeddings=True)
    d_ids  = col.query(query_embeddings=q_emb.tolist(),
                       n_results=10, include=["metadatas"])["ids"][0]
    toks   = q.lower().split()
    b_scr  = bm25.get_scores(toks)
    b_ids  = [bm25_ids[i] for i in np.argsort(b_scr)[::-1][:50]]
    h_ids  = [cid for cid, _ in rrf_fuse([d_ids, b_ids])][:10]

    for rank, cid in enumerate(h_ids, 1):
        row = chunk_lookup.get(cid, {})
        annotation_rows.append({
            "question":    q,
            "query_type":  item["type"],
            "rank":        rank,
            "chunk_id":    cid,
            "title":       row.get("title",""),
            "journal":     row.get("journal",""),
            "passage":     row.get("chunk_text","")[:600],
            "relevant":    "",   # ← Collaborator A fills this: 1 or 0
        })

annot_df = pd.DataFrame(annotation_rows)
annot_path = RESULTS_DIR / "passages_for_annotation.csv"
annot_df.to_csv(annot_path, index=False)
print(f"Annotation CSV saved: {annot_path}")
print(f"Rows: {len(annot_df):,}  (50 questions × 10 passages each)")
print("Send this to Collaborator A — they fill the 'relevant' column (1/0)")


## Step 16 — Package all results for download

In [ ]:
import shutil, zipfile

zip_path = WORK_DIR / "sciret_tier2_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RESULTS_DIR.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(WORK_DIR))

size_mb = zip_path.stat().st_size / 1e6
print(f"Results zipped → {zip_path}  ({size_mb:.1f} MB)")
print()
print("Files in results folder:")
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"  {f.name:50s} {f.stat().st_size/1e3:8.1f} KB")

print()
print("=" * 60)
print("TIER 2 COMPLETE")
print("Download sciret_tier2_results.zip from the Output panel")
print("=" * 60)
